# M6 L6: Chunking and Context Preservation

**Build three chunkers, ablate them on a fixed corpus.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

**Chunking** means cutting a long document into smaller pieces so the search engine can find the right piece quickly.

Imagine a 50-page book. If you search "vacation policy" and the search engine returns the whole book, you have to find the answer yourself. If the book is pre-cut into one-page chapters, the engine can return just the vacation page.

There are three ways to cut: **fixed window** (every 400 characters, like tearing pages evenly), **recursive** (cut at paragraph breaks first, then sentences), and **structure-aware** (keep section headings with their paragraphs). The third way is usually best for organized documents.

## The 5 things to know

1. **Chunking is the highest-leverage knob in RAG quality** — 200 vs 800 chars can mean 30 points of recall.
2. **Three families**: fixed window, recursive, structure-aware.
3. **Default pick**: recursive; upgrade to structure-aware if the docs have headings.
4. **Always ablate on your golden set** — not on the textbook.
5. **The four knobs**: chunk_size, overlap, separators, heading awareness.

In [1]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm06-chunking' / 'L6-chunking.py'
_spec = importlib.util.spec_from_file_location('L6', _p)
_L6 = importlib.util.module_from_spec(_spec)
sys.modules['L6'] = _L6
_spec.loader.exec_module(_L6)

print(f'Setup OK. Lesson module: {_L6.LESSON_NUMBER}')

Setup OK. Lesson module: M6.L6


In [2]:
# Run the 10 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L6, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 26 tests...

  PASS  test_ablation_picks_winner
  PASS  test_ablation_recall_in_range
  PASS  test_ablation_returns_three_rows
  PASS  test_ablation_zero_recall_is_handled
  PASS  test_chunk_fixed_window_chunk_id_format
  PASS  test_chunk_fixed_window_large_overlap_covers_everything
  PASS  test_chunk_id_unique
  PASS  test_chunk_recursive_chunk_id_format
  PASS  test_chunk_start_increases
  PASS  test_chunk_structure_chunk_id_format
  PASS  test_chunker_count_consistent
  PASS  test_chunker_label_matches_method
  PASS  test_chunker_latency_ms_returns_stats
  PASS  test_chunker_latency_ms_runs_all_three
  PASS  test_chunker_latency_ms_within_budget
  PASS  test_chunker_math_invariants
  PASS  test_chunks_have_doc_id
  PASS  test_fixed_window_covers_text
  PASS  test_fixed_window_handles_empty_text
  PASS  test_fixed_window_invalid_overlap
  PASS  test_recursive_handles_long_sentence
  PASS  test_recursive_handles_single_word
  PASS  test_recursive_splits_paragraphs
  PASS  test

In [3]:
# === Section 1: fixed-window chunker ===
fw = _L6.chunk_fixed_window('doc-1', _L6.SAMPLE_TEXT, chunk_size=400, overlap=50)
print(f'n_chunks: {len(fw)}')
for c in fw[:2]:
    print(f'  {c.chunk_id}  chars={len(c.text)}  text={c.text[:60]!r}...')

n_chunks: 7
  doc-1-fw000  chars=400  text='# Vacation Policy\nAll full-time employees accrue 20 days of '...
  doc-1-fw001  chars=400  text='alary and 50% of the next 2%. Vesting is immediate. Enrollme'...


In [4]:
# === Section 2: recursive chunker ===
rc = _L6.chunk_recursive('doc-1', _L6.SAMPLE_TEXT, chunk_size=400, overlap=50)
print(f'n_chunks: {len(rc)}')
for c in rc[:2]:
    print(f'  {c.chunk_id}  chars={len(c.text)}  text={c.text[:60]!r}...')

n_chunks: 9
  doc-1-rc000  chars=290  text='# Vacation Policy\nAll full-time employees accrue 20 days of '...
  doc-1-rc001  chars=401  text='# 401k Match\nThe company matches 100% of the first 3% of sal'...


In [5]:
# === Section 3: structure-aware chunker ===
st = _L6.chunk_structure('doc-1', _L6.SAMPLE_TEXT, sentences_per_chunk=5)
print(f'n_chunks: {len(st)}')
for c in st[:2]:
    print(f'  {c.chunk_id}  chars={len(c.text)}  text={c.text[:60]!r}...')

n_chunks: 10
  doc-1-st00-000  chars=289  text='Vacation Policy. All full-time employees accrue 20 days of v'...
  doc-1-st01-000  chars=195  text='401k Match. The company matches 100% of the first 3% of sala'...


In [6]:
# === Section 4: chunk count comparison ===
print('Chunk counts:\n')
print(f'  fixed_window: {len(fw)}')
print(f'  recursive:    {len(rc)}')
print(f'  structure:    {len(st)}')

Chunk counts:

  fixed_window: 7
  recursive:    9
  structure:    10


In [7]:
# === Section 5: chunk size distribution ===
print(f'{"chunker":<14} {"mean":>8} {"max":>5} {"min":>5}')
for name, chunks in (('fixed_window', fw), ('recursive', rc), ('structure', st)):
    sizes = [len(c.text) for c in chunks]
    print(f'  {name:<14} {sum(sizes)/len(sizes):>6.1f}  {max(sizes):>4}  {min(sizes):>4}')

chunker            mean   max   min
  fixed_window    373.3   400   213
  recursive       255.1   401   202
  structure       228.5   289   195


In [8]:
# === Section 6: ablation (mocked recall) ===
rt = {'fixed_window': 0.62, 'recursive': 0.78, 'structure': 0.85}
rows = _L6.ablation('doc-1', _L6.SAMPLE_TEXT, recall_table=rt)
print(f'{"chunker":<14} {"n_chunks":>9} {"mean":>8} {"max":>5} {"recall@3":>10}')
for r in rows:
    print(f'  {r.chunker:<14} {r.n_chunks:>9} {r.mean_chunk_chars:>8.1f} {r.max_chunk_chars:>5} {r.recall_at_3:>10.2f}')

chunker         n_chunks     mean   max   recall@3
  fixed_window           7    373.3   400       0.62
  recursive              9    255.1   401       0.78
  structure             10    228.5   289       0.85


In [9]:
# === Section 7: chunk size sweep on structure ===
print('Sweep sentences_per_chunk on structure chunker:\n')
for spc in (1, 3, 5, 10):
    chunks = _L6.chunk_structure('doc-1', _L6.SAMPLE_TEXT, sentences_per_chunk=spc)
    sizes = [len(c.text) for c in chunks]
    print(f'  spc={spc:>2}  n={len(chunks):>3}  mean={sum(sizes)/max(len(sizes),1):>5.0f} chars')

Sweep sentences_per_chunk on structure chunker:

  spc= 1  n= 41  mean=   66 chars
  spc= 3  n= 19  mean=  127 chars
  spc= 5  n= 10  mean=  228 chars
  spc=10  n= 10  mean=  228 chars


## Architecture (Mermaid)

```mermaid
flowchart LR
    A[document text] --> B{chunker}
    B -->|fixed window| F[split every N chars<br/>with overlap]
    B -->|recursive| R[split on paragraphs<br/>then sentences<br/>then words]
    B -->|structure-aware| S[keep heading as anchor<br/>+ N sentences]
    F --> C[Chunk objects]
    R --> C
    S --> C
    C --> E[embed chunks]
    C --> AB[ablation: compare<br/>recall@K on golden set]
    E --> VDB[vector DB]
    AB --> PICK[pick winner]
```

**Read as:** Chunk first, embed second, ablate always.

## Reflect (10 min)

1. Why is structure-aware better than fixed window for policy docs?
2. What's the trade-off between chunk size and recall? When would you choose 200 chars over 800?
3. How does overlap help with boundary straddles? What's the cost?
4. Why must you ablate on YOUR golden set, not the textbook?
5. (Stretch) Implement `chunk_size_sweep` that returns the recall curve for a range of sizes.

---

**Read next:**
- `lessons/m07-retrieval/L7-retrieval-ranking.ipynb` — BM25, dense, RRF, MMR
- `lessons/m09-rag-e2e/L9-rag-end-to-end.ipynb` — the full RAG pipeline
- `SYLLABUS.md` — the 18 modules and 21-day plan